# 01 — Data cleaning and feature engineering

Implements **Plan A** of `plan_3.md`. Each section follows the same four-part frame:

- **We have** — what the supplied file gives us.
- **The issue** — why we cannot use it as it arrives.
- **We do** — the change, and the reason.
- **New variables** — the columns the step adds.

**Output:** `data/features/panel.parquet` — one row per date and market, 2023-01-01 onward.

Run top to bottom. Every step prints its own counts and appends to `reports/cleaning_log.md`,
so the numbers quoted in the paper can always be traced back to a cell here.

## 0. Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 130)
pd.set_option("display.max_columns", 50)

PROJECT    = Path.cwd().parent.parent          # notebook lives in experiments/notebooks/
DATA_DIR   = PROJECT / "dataset" / "01a - DCT Dataset"
CLEAN_DIR  = PROJECT / "data" / "clean"
FEAT_DIR   = PROJECT / "data" / "features"
REF_DIR    = PROJECT / "reference"
REPORT_DIR = PROJECT / "reports"
for d in (CLEAN_DIR, FEAT_DIR, REF_DIR, REPORT_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Split boundaries. Plan_3 section B1. The test window is the officially held-out period.
PANEL_START = pd.Timestamp("2023-01-01")   # 2023 because 2022 is monthly-grain (step A1)
TRAIN_END   = pd.Timestamp("2024-12-31")
TUNE_END    = pd.Timestamp("2025-07-31")
TEST_END    = pd.Timestamp("2026-02-28")

assert DATA_DIR.exists(), f"dataset not found at {DATA_DIR}"
print("project:", PROJECT)

In [ ]:
# The cleaning log. Every decision we make gets one line, with its count.
LOG = []

def log(step, note):
    LOG.append({"step": step, "note": note})
    print(f"  [{step}] {note}")

def in_window(df, lo, hi):
    """Rows with Date in [lo, hi]. Always slice a frame by its OWN dates: a boolean mask
    built on one frame must never be reused on another, because row order differs."""
    return df[(df["Date"] >= lo) & (df["Date"] <= hi)]

In [ ]:
# Chart style. Palette is the validated default from the dataviz reference:
# blue is slot 1, orange slot 2, aqua slot 3. Chrome stays recessive so the data reads first.
SURFACE, INK, INK2, MUTED = "#fcfcfb", "#0b0b0b", "#52514e", "#898781"
GRID, AXIS = "#e1e0d9", "#c3c2b7"
BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"
CRITICAL = "#d03b3b"

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "font.family": "sans-serif", "font.size": 10,
    "figure.dpi": 110, "axes.titlesize": 11, "axes.titlelocation": "left",
})

def style(ax, title=None, xlabel=None, ylabel=None, grid_axis="y"):
    """Recessive grid and axes, no top/right spines, muted ticks."""
    if grid_axis:
        ax.grid(True, axis=grid_axis, color=GRID, linewidth=0.8)
        ax.set_axisbelow(True)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    for s in ("left", "bottom"):
        ax.spines[s].set_color(AXIS)
        ax.spines[s].set_linewidth(1.0)
    ax.tick_params(colors=MUTED, labelsize=9, length=0)
    if title:  ax.set_title(title, color=INK, pad=10)
    if xlabel: ax.set_xlabel(xlabel, color=INK2, fontsize=9)
    if ylabel: ax.set_ylabel(ylabel, color=INK2, fontsize=9)
    return ax

## 1. Load the raw files

Five supplied files, one sheet each (`Export`). The Excel read of `flight_data.xlsx` is slow
(15 MB, 117k rows), so we cache a raw parquet copy on first run. Delete `data/clean/*.raw.parquet`
to force a re-read from Excel.

In [ ]:
def load_raw(filename):
    """Read a supplied .xlsx, caching an untouched parquet copy for fast re-runs."""
    cache = CLEAN_DIR / (filename.replace(".xlsx", ".raw.parquet"))
    if cache.exists():
        return pd.read_parquet(cache)
    df = pd.read_excel(DATA_DIR / filename)
    df.to_parquet(cache, index=False)
    return df

flights_raw = load_raw("flight_data.xlsx")
intl_train  = load_raw("data international_train.xlsx")
intl_test   = load_raw("data international_test.xlsx")
dom_train   = load_raw("data domestic_train.xlsx")
dom_test    = load_raw("data domestic_test.xlsx")

for name, df in [("flights", flights_raw), ("intl_train", intl_train), ("intl_test", intl_test),
                 ("dom_train", dom_train), ("dom_test", dom_test)]:
    d = pd.to_datetime(df["Date"])
    print(f"{name:11} {len(df):>7,} rows  {df.shape[1]:>2} cols  "
          f"{d.min():%Y-%m-%d} to {d.max():%Y-%m-%d}")

In [ ]:
# The target column `Guests` is absent from BOTH test files. That is what we must predict.
print("intl_train columns:", list(intl_train.columns))
print("intl_test  columns:", list(intl_test.columns))
print("\nmissing from test:", sorted(set(intl_train.columns) - set(intl_test.columns)))

## 2. A1 — the flight file mixes two time grains

- **We have** — flight rows from 2022-01-01 to 2026-02-28, under one `Date` column.
- **The issue** — 2022 holds only 12 dates, one per month. Those rows are *monthly totals*.
  2023 onward is genuinely daily. One column name covers two grains, so a daily model would
  read 2022 as twelve very large days.
- **We do** — detect the grain from the data instead of hard-coding the year, then keep daily rows
  only. 2022 is also a COVID recovery year, so its levels would bias any calibration.
- **New variables** — `grain` (`monthly` or `daily`).

In [ ]:
flights = flights_raw.copy()
flights["Date"]   = pd.to_datetime(flights["Date"])
flights["market"] = flights["Departure Country Name"].str.upper().str.strip()   # step A5 key

# Data-driven grain test: a month with only one distinct date cannot be daily.
days_seen = flights.groupby(flights["Date"].dt.to_period("M"))["Date"].transform("nunique")
flights["grain"] = np.where(days_seen > 1, "daily", "monthly")

summary = (flights.groupby("grain")
           .agg(rows=("Date", "size"), dates=("Date", "nunique"),
                mean_seats_per_row=("Total Seats", "mean"),
                first=("Date", "min"), last=("Date", "max"))
           .round(0))
print(summary.to_string())
log("A1", f"grain split: {(flights['grain']=='monthly').sum():,} monthly rows (2022), "
         f"{(flights['grain']=='daily').sum():,} daily rows")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 2.8))
per_month = flights.groupby(flights["Date"].dt.to_period("M"))["Date"].nunique()
x = per_month.index.to_timestamp()
colors = [CRITICAL if v <= 1 else BLUE for v in per_month.values]
ax.bar(x, per_month.values, width=22, color=colors, linewidth=0)
ax.axhline(28, color=AXIS, linewidth=1, linestyle=(0, (4, 3)))
ax.annotate("a full month of days", xy=(x[20], 29), fontsize=8.5, color=INK2)
ax.annotate("2022: one date per month\n(monthly totals)", xy=(x[2], 4),
            fontsize=8.5, color=CRITICAL)
style(ax, "Distinct dates present per month, flight file", ylabel="dates in the file")
plt.tight_layout(); plt.show()

In [ ]:
# From here on: daily rows only.
fl = flights[flights["grain"] == "daily"].copy()
log("A1", f"model table restricted to daily rows from {fl['Date'].min():%Y-%m-%d}")

## 3. A2 — the frequency column is not a weekly count

- **We have** — `Average Weekly Frequency`, one value per route and day. Example values: 0.2258,
  0.4516, 0.6774.
- **The issue** — the name says weekly, but the number is neither a weekly nor a daily count.
  Nobody can set a frequency slider from it. It is also empty for every 2022 row.
- **We do** — decode it. `flights that day = AWF x days_in_month / 7`. We assert the result is a
  whole number, which is the proof the decoding is right.
- **New variables** — `flights_per_day`, `weekly_frequency`, `seats_per_aircraft`.

In [ ]:
dim = fl["Date"].dt.days_in_month
raw_flights = fl["Average Weekly Frequency"] * dim / 7

# The proof: if the decoding is correct, every value is a whole number.
worst = (raw_flights - raw_flights.round()).abs().max()
print(f"largest distance from a whole number: {worst:.2e}  (over {raw_flights.notna().sum():,} rows)")
assert worst < 1e-6, "frequency decoding is wrong"

fl["flights_per_day"]    = raw_flights.round().astype("int64")
fl["seats_per_aircraft"] = fl["Total Seats"] / fl["flights_per_day"].replace(0, np.nan)

# A route's true average weekly frequency = the sum of AWF across that month's days.
fl["ym"] = fl["Date"].dt.to_period("M")
fl["weekly_frequency"] = (fl.groupby(["ym", "Departure City", "Airline Name"])
                            ["Average Weekly Frequency"].transform("sum"))

log("A2", f"decoded flights_per_day for {len(fl):,} rows, max integer error {worst:.1e}")

In [ ]:
# Worked example from plan_3: Yerevan, January 2024.
ex = fl[(fl["ym"] == pd.Period("2024-01")) & (fl["Departure City"] == "Yerevan")]
print(ex[["Date", "Airline Name", "Average Weekly Frequency", "flights_per_day",
          "Total Seats", "seats_per_aircraft"]].head(6).to_string(index=False))
print(f"\nmonth totals -> {ex['flights_per_day'].sum()} flights, "
      f"{ex['weekly_frequency'].iloc[0]:.2f} per week, "
      f"{ex['Total Seats'].sum() / ex['flights_per_day'].sum():.0f} seats per aircraft")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.4))
spa = fl["seats_per_aircraft"].dropna()
counts = spa.round().value_counts()

heights, edges, _ = ax.hist(spa[spa < 450], bins=np.arange(100, 452, 2), color=BLUE, linewidth=0)

# Label each gauge above its own spike. Take the height from the histogram itself, not from a
# value count: a 2-seat bin can hold more than one gauge, so the bar is the taller of the two.
# Neighbouring gauges (174/186, 230/239) sit a few seats apart, so labels alternate height.
for i, gauge in enumerate((174, 186, 230, 239, 290, 327, 371)):
    b = np.clip(np.searchsorted(edges, gauge, side="right") - 1, 0, len(heights) - 1)
    ax.annotate(str(gauge), xy=(gauge, heights[b]), xytext=(0, 20 if i % 2 == 0 else 38),
                textcoords="offset points", ha="center", fontsize=8.5, color=INK2,
                arrowprops=dict(arrowstyle="-", color=AXIS, linewidth=0.8, shrinkA=0, shrinkB=2))
ax.margins(y=0.26)
style(ax, "Decoded seats per aircraft lands on real airframe gauges",
      xlabel="seats per aircraft", ylabel="route-days")
plt.tight_layout(); plt.show()
print("most common gauges:", counts.head(8).index.astype(int).tolist())

## 4. A3 — some load factors are above 1

- **We have** — `Load Factor`.
- **The issue** — it equals `Total PAX / Total Seats` exactly, and some rows report more passengers
  than seats. A real flight cannot do that.
- **We do** — keep the supplied column untouched and add a capped copy beside it, plus a flag. We
  never change a supplied number in place.
- **New variables** — `load_factor_clean` (capped at 1.00), `load_factor_capped` (true/false).

In [ ]:
# Identity check first: the column is a pure division, so capping it is safe and reversible.
identity_err = (fl["Load Factor"] - fl["Total PAX"] / fl["Total Seats"]).abs().max()
print(f"max |Load Factor - PAX/Seats| = {identity_err:.2e}")
assert identity_err < 1e-9

over_1    = int((fl["Load Factor"] > 1).sum())
over_105  = int((fl["Load Factor"] > 1.05).sum())
fl["load_factor_capped"] = fl["Load Factor"] > 1
fl["load_factor_clean"]  = fl["Load Factor"].clip(upper=1.0)

print(f"rows above 1.00: {over_1:,} ({over_1/len(fl):.1%})   above 1.05: {over_105:,}   "
      f"max: {fl['Load Factor'].max():.3f}")
log("A3", f"capped {over_1:,} rows ({over_1/len(fl):.1%}) at load factor 1.00, "
         f"of which only {over_105:,} exceeded 1.05")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.0))
ax.hist(fl["Load Factor"].clip(upper=1.4), bins=140, color=BLUE, linewidth=0)
ax.axvspan(1.0, 1.4, color=CRITICAL, alpha=0.10, linewidth=0)
ax.axvline(1.0, color=CRITICAL, linewidth=1.5)
ax.annotate(f"{over_1:,} route-days above 1.00\n(only {over_105} above 1.05)",
            xy=(1.02, 0.80), xycoords=("data", "axes fraction"),
            fontsize=8.5, color=CRITICAL)
style(ax, "Load factor: the overshoot is real but almost entirely small",
      xlabel="load factor (clipped at 1.4 for display)", ylabel="route-days")
plt.tight_layout(); plt.show()

## 5. A4 — half the arriving passengers never reach a hotel

- **We have** — `Total PAX`, split into `Total P2P`, `Total Transfer` and `Total Transit`.
- **The issue** — transfer passengers change aircraft and leave. A model built on `Total PAX`
  counts roughly twice the real hotel demand.
- **We do** — use `Total P2P` as the passenger base, and keep the transfer share as its own
  variable, because it differs enormously by market.
- **New variables** — `transfer_share`.

In [ ]:
parts = fl["Total P2P"] + fl["Total Transfer"] + fl["Total Transit"]
print(f"rows where P2P + Transfer + Transit == Total PAX: {(parts == fl['Total PAX']).sum():,}"
      f" / {len(fl):,}")
assert (parts == fl["Total PAX"]).all(), "passenger decomposition does not hold"

fl["transfer_share"] = fl["Total Transfer"] / fl["Total PAX"].replace(0, np.nan)

share_all = fl["Total Transfer"].sum() / fl["Total PAX"].sum()
transit   = fl["Total Transit"].sum() / fl["Total PAX"].sum()
print(f"\ntransfer share of all arrivals: {share_all:.1%}    transit: {transit:.4%}")
log("A4", f"using Total P2P as the passenger base; transfer is {share_all:.1%} of arrivals")

In [ ]:
by_mkt = (fl.groupby("market")
            .apply(lambda g: g["Total Transfer"].sum() / g["Total PAX"].sum(), include_groups=False)
            .sort_values())

fig, ax = plt.subplots(figsize=(8, 7))
ax.barh(by_mkt.index, by_mkt.values, color=BLUE, height=0.72, linewidth=0)
ax.axvline(share_all, color=ORANGE, linewidth=1.5)
ax.annotate(f"overall {share_all:.0%}", xy=(share_all + 0.01, 0.5), fontsize=8.5, color=ORANGE)
ax.set_xlim(0, 1)
style(ax, "Share of arrivals that are transfer passengers, by market",
      xlabel="transfer share of Total PAX", grid_axis="x")
plt.tight_layout(); plt.show()

## 6. A5 — flight country and hotel country mean different things

- **We have** — `Departure Country Name` in the flight file (33 values) and `Nationality` in the
  hotel file (45 values).
- **The issue** — one names where the aircraft took off from. The other names the passport the
  guest holds. One plane load of people becomes many passport types in the hotels.
- **We do** — normalise both keys to a single `market` column, then join the exact matches. Measure
  the capture ratio per market so the mismatch is visible as a number, not an assumption.
- **New variables** — `market`, `has_direct_route`.

In [ ]:
intl = pd.concat([intl_train.assign(src="train"), intl_test.assign(src="test")], ignore_index=True)
intl["Date"]   = pd.to_datetime(intl["Date"])
intl["market"] = intl["Nationality"].str.upper().str.strip()

flight_markets = set(fl["market"].unique())
hotel_markets  = set(intl["market"].unique())
no_direct      = sorted(hotel_markets - flight_markets)

print(f"flight markets: {len(flight_markets)}   hotel markets: {len(hotel_markets)}")
print(f"flight-only (should be empty): {sorted(flight_markets - hotel_markets)}")
print(f"\nno direct route ({len(no_direct)}):")
print("  " + ", ".join(m.title() for m in no_direct))
assert not (flight_markets - hotel_markets), "a flight market is missing from the hotel file"
log("A5", f"{len(flight_markets)} flight markets are a strict subset of {len(hotel_markets)} "
         f"hotel markets; {len(no_direct)} markets have no direct route")

In [ ]:
# Capture ratio = hotel arrivals of a passport / direct P2P arrivals from that country.
# Computed on TRAIN ONLY, so nothing downstream can leak the tune or test window.
fl_m = (fl[fl["Date"] <= TRAIN_END].groupby(["ym", "market"], as_index=False)["Total P2P"].sum())
ho_m = (intl[intl["Date"] <= TRAIN_END].assign(ym=lambda d: d["Date"].dt.to_period("M"))
            .groupby(["ym", "market"], as_index=False)["New Arrivals"].sum())

cap = fl_m.merge(ho_m, on=["ym", "market"])
cap = cap[cap["Total P2P"] > 0]
cap["capture"] = cap["New Arrivals"] / cap["Total P2P"]

capture_by_market = cap.groupby("market")["capture"].median().sort_values()
above_one = (cap["capture"] > 1).sum()
print(f"market-months with capture above 1: {above_one} / {len(cap)}")
print(f"\nhighest: {capture_by_market.tail(3).round(2).to_dict()}")
print(f"lowest:  {capture_by_market.head(3).round(2).to_dict()}")
log("A5", f"capture ratio ranges {capture_by_market.min():.2f} to {capture_by_market.max():.2f} "
         f"across markets; {above_one}/{len(cap)} market-months exceed 1.0")

In [ ]:
# A dot plot, not bars: the scale is logarithmic and the meaning is distance from 1.0, so
# position carries it. A bar's length would be measured from an arbitrary left edge.
# The ratio diverges around 1.0, so the two sides take the diverging pair, blue and red.
fig, ax = plt.subplots(figsize=(8, 7))
vals = capture_by_market.values
names = list(capture_by_market.index)
above = vals > 1.0

ax.hlines(names, 1.0, vals, color=[CRITICAL if a else BLUE for a in above],
          linewidth=1.5, alpha=0.45)
ax.scatter(vals[~above], np.array(names)[~above], s=46, color=BLUE, zorder=3,
           edgecolor=SURFACE, linewidth=1.5, label="mostly arrives on direct flights (< 1)")
ax.scatter(vals[above], np.array(names)[above], s=46, color=CRITICAL, zorder=3,
           edgecolor=SURFACE, linewidth=1.5, label="mostly arrives some other way (> 1)")
ax.axvline(1.0, color=INK2, linewidth=1.2)
ax.set_xscale("log")
ax.set_xticks([0.05, 0.1, 0.25, 0.5, 1, 2, 4, 8])
ax.set_xticklabels(["0.05", "0.1", "0.25", "0.5", "1", "2", "4", "8"])
ax.annotate("1.0 — what a single global\ncapture rate assumes for everyone",
            xy=(0.03, 0.72), xycoords="axes fraction", fontsize=8.5, color=INK2)
leg = ax.legend(frameon=False, loc="upper left", fontsize=8.5,
                bbox_to_anchor=(0.02, 0.98))
for t in leg.get_texts():
    t.set_color(INK2)
style(ax, "Capture ratio by market: hotel arrivals per direct P2P arrival",
      xlabel="median ratio, train window (log scale)", grid_axis="x")
plt.tight_layout(); plt.show()

In [ ]:
# reference/bridge.csv — the inspectable origin-to-nationality assumption.
# Direct matches get weight 1.0. The 12 markets with no route need an indirect allocation,
# which is a modelling decision (plan_3 step B5), so we record them as pending rather than
# inventing a number here.
bridge = pd.DataFrame({"market": sorted(hotel_markets)})
bridge["has_direct_route"] = bridge["market"].isin(flight_markets)
bridge["direct_weight"]    = np.where(bridge["has_direct_route"], 1.0, 0.0)
bridge["capture_rate_train"] = bridge["market"].map(capture_by_market).round(4)
bridge["allocation"] = np.where(bridge["has_direct_route"], "direct match",
                                "pending: similarity prior (plan_3 B5)")
bridge.to_csv(REF_DIR / "bridge.csv", index=False)
print(f"wrote {REF_DIR / 'bridge.csv'}  ({len(bridge)} rows)")
bridge.head(8)

## 7. A6 — blank cells mean two different things

- **We have** — hotel rows with empty cells, and an incomplete date-by-market grid.
- **The issue** — a blank can mean zero, or it can mean unknown. A zero in a ratio gives a wrong
  answer. A missing value in a ratio gives no answer. The two need different rules.
- **We do** — apply one written rule per column, complete the grid, and flag every cell we touched.
- **New variables** — `was_filled`.

In [ ]:
print("blanks in the international hotel data:")
print(intl.isna().sum().to_string())
print(f"\ngrid: {intl['Date'].nunique()} dates x {intl['market'].nunique()} markets = "
      f"{intl['Date'].nunique() * intl['market'].nunique():,} cells, but {len(intl):,} rows")

In [ ]:
# Rule 1: a missing Same-Day Guests count means no such guest that day -> 0.
sdg_blanks = int(intl["Same-Day Guests"].isna().sum())
intl["Same-Day Guests"] = intl["Same-Day Guests"].fillna(0)

# Rule 2: a missing New Arrivals value is unknown, not zero. Leave it as NaN so it stays
# out of every ratio. (Guests is legitimately absent for the whole test window.)
na_blanks = int(intl["New Arrivals"].isna().sum())

log("A6", f"filled {sdg_blanks:,} blank Same-Day Guests with 0")
log("A6", f"left {na_blanks:,} blank New Arrivals as missing, so they stay out of ratios")
print(f"Same-Day Guests filled: {sdg_blanks:,}      New Arrivals left missing: {na_blanks:,}")

In [ ]:
# Complete the date x market grid over the panel window, and mark what we added.
panel_dates = pd.date_range(PANEL_START, TEST_END, freq="D")
grid = pd.MultiIndex.from_product([panel_dates, sorted(hotel_markets)], names=["Date", "market"])

hotel = (intl.set_index(["Date", "market"])[["Guests", "New Arrivals", "Same-Day Guests"]]
             .reindex(grid).reset_index())
hotel["was_filled"] = hotel["Same-Day Guests"].isna()          # rows absent from the source
hotel["Same-Day Guests"] = hotel["Same-Day Guests"].fillna(0)

n_filled = int(hotel["was_filled"].sum())
print(f"grid rows: {len(hotel):,}   added by completion: {n_filled:,}")
log("A6", f"completed the date x market grid, adding {n_filled:,} rows flagged was_filled")

In [ ]:
# Rule 3: a completed grid cell gets Guests = 0, but ONLY after we check the nearby days are
# small. A gap on a busy day would be a real data loss, and zero would be a lie. We run the
# check in code rather than asserting it, and we print what it found.
#
# Note the scope: only dates up to TUNE_END. Over the test window `Guests` is absent by design,
# because that is the quantity we must predict. Those stay NaN.
gaps = hotel["was_filled"] & (hotel["Date"] <= TUNE_END) & hotel["Guests"].isna()

h_sorted = hotel.sort_values(["market", "Date"])
hotel["nbr_mean"] = (h_sorted.groupby("market")["Guests"]
                     .transform(lambda s: s.rolling(7, center=True, min_periods=1).mean()))
mkt_median = hotel.groupby("market")["Guests"].transform("median")
is_small = hotel["nbr_mean"] < 0.5 * mkt_median        # neighbourhood below half the usual level

print(f"grid gaps needing a value: {int(gaps.sum())}")
if gaps.any():
    report = (hotel[gaps].assign(small=is_small[gaps])
              .groupby("market")
              .agg(gaps=("Date", "size"), nbr_mean=("nbr_mean", "mean"),
                   passed_check=("small", "sum")))
    report["market_median"] = mkt_median.groupby(hotel["market"]).first().reindex(report.index)
    print(report.round(1).to_string())

loud = gaps & ~is_small
if loud.any():
    print(f"\n{int(loud.sum())} gap(s) sit on NORMAL days - not filling those, "
          f"they stay missing:")
    print(hotel.loc[loud, ["Date", "market", "nbr_mean"]].to_string(index=False))

fillable = gaps & is_small
hotel.loc[fillable, "Guests"] = 0.0
hotel = hotel.drop(columns=["nbr_mean"])
log("A6", f"set Guests = 0 on {int(fillable.sum())} completed grid cells whose 7-day "
         f"neighbourhood was below half the market median; left {int(loud.sum())} as missing")

## 8. A7 — flights and guest nights do not fall on the same day

- **We have** — daily flight arrivals, and daily guest nights.
- **The issue** — a guest who lands today still holds a bed for several nights. Same-day flights
  therefore explain today's guest nights poorly.
- **We do** — aggregate flights to date x market, complete that grid too (so a rolling window
  counts *days*, not rows), then build trailing sums. We shift nothing forward: flights come
  before guests, never after.
- **New variables** — `p2p_3d`, `p2p_7d`, `p2p_14d`, `p2p_28d`, `seats_7d`, `flights_7d`.

In [ ]:
# One row per date and market, summed over every airline and departure city.
fm = (fl.groupby(["Date", "market"], as_index=False)
        .agg(p2p=("Total P2P", "sum"), pax=("Total PAX", "sum"), seats=("Total Seats", "sum"),
             transfer=("Total Transfer", "sum"), flights=("flights_per_day", "sum"),
             n_airlines=("Airline Name", "nunique"), n_cities=("Departure City", "nunique")))

print(f"{len(fm):,} market-days, {fm['market'].nunique()} markets")
fm.head(4)

In [ ]:
# A market's average weekly frequency for a month = the sum of AWF over that month's days.
wk = (fl.groupby(["ym", "market"], as_index=False)["Average Weekly Frequency"]
        .sum().rename(columns={"Average Weekly Frequency": "weekly_frequency"}))
print(wk.sort_values("weekly_frequency", ascending=False).head(5).to_string(index=False))

In [ ]:
# Complete the market-day grid BEFORE rolling, so a 7-day window always means 7 calendar days.
fm_full = (fm.set_index(["Date", "market"]).reindex(grid).reset_index())
flight_cols = ["p2p", "pax", "seats", "transfer", "flights", "n_airlines", "n_cities"]
fm_full[flight_cols] = fm_full[flight_cols].fillna(0)
fm_full = fm_full.sort_values(["market", "Date"]).reset_index(drop=True)

g = fm_full.groupby("market", sort=False)
for w in (3, 7, 14, 28):
    fm_full[f"p2p_{w}d"] = g["p2p"].transform(lambda s, w=w: s.rolling(w, min_periods=1).sum())
for col in ("seats", "flights", "pax", "transfer"):
    fm_full[f"{col}_7d"] = g[col].transform(lambda s: s.rolling(7, min_periods=1).sum())

# Build the supply RATIOS over the same 7-day window, not over a single day. Small markets do
# not fly daily, so a day-level ratio is 0/0 on every quiet day. Over a week the ratio is both
# defined far more often and more meaningful: it describes the flights that brought the guests
# who are in beds right now.
fm_full["load_factor_clean"]  = (fm_full["pax_7d"] / fm_full["seats_7d"].replace(0, np.nan)).clip(upper=1.0)
fm_full["transfer_share"]     = fm_full["transfer_7d"] / fm_full["pax_7d"].replace(0, np.nan)
fm_full["seats_per_aircraft"] = fm_full["seats_7d"] / fm_full["flights_7d"].replace(0, np.nan)

log("A7", "built trailing 3/7/14/28-day sums on a completed market-day grid")
log("A7", f"supply ratios use the 7-day window: that leaves "
         f"{int(fm_full['load_factor_clean'].isna().sum()):,} undefined market-days "
         f"instead of {int((fm_full['seats'] == 0).sum()):,} at day level")
fm_full[["Date", "market", "p2p", "p2p_7d", "seats_7d", "load_factor_clean"]].tail(3)

In [ ]:
# Why 7 days: correlate daily guest nights against each trailing window.
daily_guests = (hotel[hotel["Date"] <= TUNE_END].groupby("Date")["Guests"].sum())
daily_p2p    = (fm_full[fm_full["Date"] <= TUNE_END].groupby("Date")["p2p"].sum())
joined = pd.concat([daily_guests, daily_p2p], axis=1).dropna()

windows = [1, 3, 5, 7, 10, 14, 21, 28]
corrs = [joined["p2p"].rolling(w).sum().corr(joined["Guests"]) for w in windows]

fig, ax = plt.subplots(figsize=(8, 3.2))
ax.plot(windows, corrs, color=BLUE, linewidth=2, marker="o", markersize=5)
best = int(np.argmax(corrs))
ax.plot(windows[best], corrs[best], marker="o", markersize=10, color=BLUE,
        markeredgecolor=SURFACE, markeredgewidth=2)
ax.annotate(f"{windows[best]} days, r = {corrs[best]:.3f}",
            xy=(windows[best] + 0.6, corrs[best] - 0.004), fontsize=9, color=INK)
style(ax, "A trailing 7-day window of arrivals explains guest nights best",
      xlabel="length of trailing window (days)", ylabel="correlation with daily guest nights")
plt.tight_layout(); plt.show()
print(dict(zip(windows, np.round(corrs, 3))))

## 9. A8 — the files hold no calendar and no season measure

- **We have** — a `Date` column. The brief lists an event calendar as supplied data, but the folder
  holds none.
- **The issue** — a month number cannot hold Ramadan, because Ramadan moves about 11 days earlier
  each year. Nothing in the files states which months are strong or weak either.
- **We do** — write `reference/events.csv` ourselves, with a `source` column on every row so a
  reader can check each date. Compute the season index from the train window only.
- **New variables** — `month`, `day_of_week`, `week_of_year`, `season_index`, and one flag per event.

> **These dates need confirming.** They are our best estimates, not an official calendar. Every row
> is marked `estimate - confirm with DCT` in the `source` column. Asking the DCT mentors for the
> official calendar is an open item in `plan.md`.

In [ ]:
# Each row: event, start, end (inclusive), source. Written out so a reader can audit it.
EVENTS = [
    # Ramadan (approximate Gregorian spans)
    ("ramadan", "2022-04-02", "2022-05-01"), ("ramadan", "2023-03-23", "2023-04-20"),
    ("ramadan", "2024-03-11", "2024-04-09"), ("ramadan", "2025-03-01", "2025-03-30"),
    ("ramadan", "2026-02-18", "2026-03-19"),
    # Eid al-Fitr, then Eid al-Adha (3-day holiday spans)
    ("eid", "2022-05-02", "2022-05-04"), ("eid", "2023-04-21", "2023-04-23"),
    ("eid", "2024-04-10", "2024-04-12"), ("eid", "2025-03-31", "2025-04-02"),
    ("eid", "2026-03-20", "2026-03-22"),
    ("eid", "2022-07-09", "2022-07-11"), ("eid", "2023-06-28", "2023-06-30"),
    ("eid", "2024-06-16", "2024-06-18"), ("eid", "2025-06-06", "2025-06-08"),
    # Abu Dhabi Grand Prix race weekends
    ("f1_weekend", "2022-11-18", "2022-11-20"), ("f1_weekend", "2023-11-24", "2023-11-26"),
    ("f1_weekend", "2024-12-06", "2024-12-08"), ("f1_weekend", "2025-12-05", "2025-12-07"),
    # UAE National Day
    ("national_day", "2022-12-02", "2022-12-03"), ("national_day", "2023-12-02", "2023-12-03"),
    ("national_day", "2024-12-02", "2024-12-03"), ("national_day", "2025-12-02", "2025-12-03"),
    # Source-market holidays
    ("chinese_new_year", "2022-01-31", "2022-02-06"), ("chinese_new_year", "2023-01-21", "2023-01-27"),
    ("chinese_new_year", "2024-02-10", "2024-02-16"), ("chinese_new_year", "2025-01-29", "2025-02-04"),
    ("chinese_new_year", "2026-02-17", "2026-02-23"),
    ("diwali", "2022-10-24", "2022-10-26"), ("diwali", "2023-11-12", "2023-11-14"),
    ("diwali", "2024-10-31", "2024-11-02"), ("diwali", "2025-10-20", "2025-10-22"),
    ("uk_half_term", "2023-02-13", "2023-02-17"), ("uk_half_term", "2023-10-23", "2023-10-27"),
    ("uk_half_term", "2024-02-12", "2024-02-16"), ("uk_half_term", "2024-10-28", "2024-11-01"),
    ("uk_half_term", "2025-02-17", "2025-02-21"), ("uk_half_term", "2025-10-27", "2025-10-31"),
    ("uk_half_term", "2026-02-16", "2026-02-20"),
]
events = pd.DataFrame(EVENTS, columns=["event", "start", "end"])
events["source"] = "estimate - confirm with DCT"
events.to_csv(REF_DIR / "events.csv", index=False)
print(f"wrote {REF_DIR / 'events.csv'}  ({len(events)} spans, "
      f"{events['event'].nunique()} event types)")
log("A8", f"wrote {len(events)} event spans to reference/events.csv, all flagged for confirmation")
events.groupby("event").size().rename("spans").to_frame()

In [ ]:
cal = pd.DataFrame({"Date": panel_dates})
cal["month"]        = cal["Date"].dt.month
cal["day_of_week"]  = cal["Date"].dt.dayofweek
cal["week_of_year"] = cal["Date"].dt.isocalendar().week.astype("int64")

for event, spans in events.groupby("event"):
    hit = np.zeros(len(cal), dtype=bool)
    for _, r in spans.iterrows():
        hit |= cal["Date"].between(pd.Timestamp(r["start"]), pd.Timestamp(r["end"])).to_numpy()
    cal[f"is_{event}"] = hit

print(cal.filter(like="is_").sum().to_string())

In [ ]:
def season_index(df, value_col, group_cols):
    """De-trended monthly demand index. Normalise each year by its own mean, then average
    across years, so growth across the window cannot masquerade as seasonality."""
    d = df.copy()
    d["year"]  = d["Date"].dt.year
    d["month"] = d["Date"].dt.month
    m = d.groupby(group_cols + ["year", "month"], as_index=False)[value_col].sum()
    m["year_mean"] = m.groupby(group_cols + ["year"])[value_col].transform("mean")
    m["idx"] = m[value_col] / m["year_mean"]
    return m.groupby(group_cols + ["month"], as_index=False)["idx"].mean()

# Train window only: 2023 and 2024 are both complete years, so no month is averaged over more
# years than another, and no tune or test information can reach the feature.
train_hotel = in_window(hotel, PANEL_START, TRAIN_END)
season_mkt  = season_index(train_hotel, "Guests", ["market"]).rename(columns={"idx": "season_index"})
print(f"season_index rows: {len(season_mkt)}  ({season_mkt['market'].nunique()} markets x 12 months)")
season_mkt.head(4)

In [ ]:
# International is a winter market, domestic is a summer market. Two series, so a legend.
dom = pd.concat([dom_train.assign(src="train"), dom_test.assign(src="test")], ignore_index=True)
dom["Date"] = pd.to_datetime(dom["Date"])

intl_idx = season_index(train_hotel.assign(k=1), "Guests", ["k"]).set_index("month")["idx"]
dom_idx  = season_index(in_window(dom, PANEL_START, TRAIN_END).assign(k=1),
                        "Guests", ["k"]).set_index("month")["idx"]

fig, ax = plt.subplots(figsize=(8.5, 3.4))
months = range(1, 13)
ax.plot(months, intl_idx.reindex(months), color=BLUE, linewidth=2, marker="o",
        markersize=5, label="International")
ax.plot(months, dom_idx.reindex(months), color=ORANGE, linewidth=2, marker="o",
        markersize=5, label="Domestic")
ax.axhline(1.0, color=AXIS, linewidth=1, linestyle=(0, (4, 3)))
ax.set_xticks(list(months))
ax.set_xticklabels(["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"])
leg = ax.legend(frameon=False, loc="lower left", fontsize=9)
for t in leg.get_texts():
    t.set_color(INK2)
style(ax, "De-trended seasonality: the two streams peak in opposite halves of the year",
      ylabel="index (year mean = 1.00)")
plt.tight_layout(); plt.show()
print("shape correlation:", round(np.corrcoef(intl_idx.reindex(months),
                                              dom_idx.reindex(months))[0, 1], 3))

## 10. A9 — the market name tells the model nothing about a new market

- **We have** — a market name, which a model reads as a label.
- **The issue** — a label carries no meaning for a market the model never saw, and question 1 asks
  exactly that. Markets also differ enormously in how strongly they answer a change in flights.
- **We do** — describe each market by numbers instead of by name. All history variables come from
  the train window only.
- **New variables** — `region`, `haul_band`, `capture_rate_hist`, `stay_length_hist`,
  `elasticity_tier`, `n_airlines`, `n_cities`, `route_is_new`, `months_since_first_flight`.

> `haul_band` is a stated approximation from `region`. A true distance band needs a coordinate per
> departure city, which the flight file does not supply — that is `reference/cities.csv` in
> `plan_3.md` and it is not built yet.

In [ ]:
REGION = {
    "BAHRAIN": "GCC", "KUWAIT": "GCC", "OMAN": "GCC", "QATAR": "GCC", "SAUDI ARABIA": "GCC",
    "EGYPT": "MENA", "ISRAEL": "MENA", "JORDAN": "MENA", "LEBANON": "MENA",
    "MOROCCO": "MENA", "TURKEY": "MENA",
    "INDIA": "South Asia", "PAKISTAN": "South Asia",
    "ARMENIA": "Caucasus/C.Asia", "AZERBAIJAN": "Caucasus/C.Asia",
    "KAZAKHSTAN": "Caucasus/C.Asia", "UZBEKISTAN": "Caucasus/C.Asia",
    "CHINA": "East Asia", "JAPAN": "East Asia", "SOUTH KOREA": "East Asia",
    "PHILIPPINES": "East Asia",
    "AUSTRIA": "W.Europe", "BELGIUM": "W.Europe", "FRANCE": "W.Europe", "GERMANY": "W.Europe",
    "IRELAND": "W.Europe", "ITALY": "W.Europe", "NETHERLANDS": "W.Europe", "SPAIN": "W.Europe",
    "SWITZERLAND": "W.Europe", "UNITED KINGDOM": "W.Europe",
    "DENMARK": "Nordics", "FINLAND": "Nordics", "NORWAY": "Nordics", "SWEDEN": "Nordics",
    "CZECHIA": "E.Europe", "POLAND": "E.Europe", "ROMANIA": "E.Europe",
    "RUSSIAN FEDERATION": "E.Europe",
    "CANADA": "N.America", "UNITED STATES OF AMERICA": "N.America", "MEXICO": "N.America",
    "BRAZIL": "LatAm", "SOUTH AFRICA": "Africa", "AUSTRALIA": "Oceania",
}
HAUL = {"GCC": "short", "MENA": "short", "South Asia": "medium", "Caucasus/C.Asia": "medium",
        "East Asia": "long", "W.Europe": "long", "Nordics": "long", "E.Europe": "long",
        "Africa": "long", "N.America": "ultra", "LatAm": "ultra", "Oceania": "ultra"}

markets = pd.DataFrame({"market": sorted(hotel_markets)})
markets["region"] = markets["market"].map(REGION)
markets["haul_band"] = markets["region"].map(HAUL)
assert markets["region"].notna().all(), markets[markets["region"].isna()]["market"].tolist()
markets.to_csv(REF_DIR / "markets.csv", index=False)
print(f"wrote {REF_DIR / 'markets.csv'}  ({len(markets)} markets, "
      f"{markets['region'].nunique()} regions)")
markets.groupby(["haul_band", "region"]).size().rename("markets").to_frame()

In [ ]:
# Train-window history per market: capture rate and stay length. Train only.
hist = (in_window(hotel, PANEL_START, TRAIN_END).groupby("market", as_index=False)
            .agg(guests=("Guests", "sum"), arrivals=("New Arrivals", "sum")))
hist["stay_length_hist"] = (hist["guests"] / hist["arrivals"]).round(3)

markets = markets.merge(hist[["market", "stay_length_hist"]], on="market", how="left")
markets["capture_rate_hist"] = markets["market"].map(capture_by_market).round(4)
markets["has_direct_route"]  = markets["market"].isin(flight_markets)

print(markets[["market", "region", "haul_band", "stay_length_hist",
               "capture_rate_hist", "has_direct_route"]].head(8).to_string(index=False))
print(f"\nstay length: median {markets['stay_length_hist'].median():.2f} nights per arrival")

In [ ]:
# Route age. Measured against the whole flight file, 2022 rows included, so it is as long a
# history as the data allows.
FILE_START   = flights["Date"].min()
first_flight = flights.groupby("market")["Date"].min()
launched_later = first_flight[first_flight > FILE_START]

print(f"file starts {FILE_START:%Y-%m-%d}")
print(f"markets whose first flight is later than that: {len(launched_later)} of {len(first_flight)}")
if len(launched_later):
    print(launched_later.sort_values().dt.strftime("%Y-%m-%d").to_string())
log("A9", f"{len(launched_later)} of {len(first_flight)} markets gained their first flight after "
         f"the file starts; the other {len(first_flight) - len(launched_later)} are present "
         f"throughout, so route-launch evidence is thin and question 1 still needs the B5 prior")

In [ ]:
# elasticity_tier: slope of log(guest nights) on log(P2P), per market, after we remove that
# market's own month-of-year mean. Train window only. This is result 3.1 of plan_3.
mm = (in_window(fm_full, PANEL_START, TRAIN_END).assign(ym=lambda d: d["Date"].dt.to_period("M"))
        .groupby(["ym", "market"], as_index=False)["p2p"].sum())
hh = (in_window(hotel, PANEL_START, TRAIN_END).assign(ym=lambda d: d["Date"].dt.to_period("M"))
        .groupby(["ym", "market"], as_index=False)["Guests"].sum())
panel_m = mm.merge(hh, on=["ym", "market"])
panel_m = panel_m[(panel_m["p2p"] > 0) & (panel_m["Guests"] > 0)].copy()
panel_m["lp"] = np.log(panel_m["p2p"])
panel_m["lg"] = np.log(panel_m["Guests"])
panel_m["moy"] = panel_m["ym"].dt.month

rows = []
for mkt, grp in panel_m.groupby("market"):
    if len(grp) < 12:
        continue
    d = grp.copy()
    for c in ("lp", "lg"):                      # remove this market's month-of-year mean
        d[c] = d[c] - d.groupby("moy")[c].transform("mean")
    if d["lp"].std() < 1e-9:
        continue
    rows.append({"market": mkt, "n": len(d),
                 "elasticity": round(float(np.polyfit(d["lp"], d["lg"], 1)[0]), 3)})
elas = pd.DataFrame(rows).sort_values("elasticity", ascending=False)

elas["elasticity_tier"] = pd.cut(elas["elasticity"], [-np.inf, 0.3, 0.7, np.inf],
                                 labels=["weak", "mixed", "strong"]).astype("object")
markets = markets.merge(elas[["market", "elasticity", "elasticity_tier"]], on="market", how="left")

# The 12 markets with no direct route have no elasticity to measure. Say so, rather than
# letting a silent NaN become a quiet zero later.
markets["elasticity_tier"] = markets["elasticity_tier"].fillna("unmeasured")

print(f"{len(elas)} markets measured   median elasticity {elas['elasticity'].median():.2f}   "
      f"positive {int((elas['elasticity'] > 0).sum())}/{len(elas)}")
print(markets["elasticity_tier"].value_counts().to_string())
log("A9", f"measured elasticity for {len(elas)} markets on train, median "
         f"{elas['elasticity'].median():.2f}, all positive: {bool((elas['elasticity']>0).all())}")

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 6.8))
e = elas.sort_values("elasticity")
tier_color = {"strong": BLUE, "mixed": ORANGE, "weak": AQUA}
ax.barh(e["market"], e["elasticity"],
        color=[tier_color[t] for t in e["elasticity_tier"]], height=0.72, linewidth=0)
ax.axvline(0, color=AXIS, linewidth=1)
med = e["elasticity"].median()
ax.axvline(med, color=INK2, linewidth=1, linestyle=(0, (4, 3)))
ax.annotate(f"median {med:.2f}", xy=(med + 0.02, 0.03), xycoords=("data", "axes fraction"),
            fontsize=8.5, color=INK2)

# A legend, so the tier is never carried by colour alone. The market names on the y-axis
# are the visible labels the contrast rule asks for.
handles = [plt.Rectangle((0, 0), 1, 1, color=tier_color[t]) for t in ("strong", "mixed", "weak")]
leg = ax.legend(handles, ["strong (> 0.7)", "mixed (0.3 - 0.7)", "weak (< 0.3)"],
                frameon=False, loc="lower right", fontsize=8.5)
for t in leg.get_texts():
    t.set_color(INK2)
style(ax, "Elasticity of guest nights to arrivals, after removing each market's own seasonality",
      xlabel="log-log slope, train window", grid_axis="x")
plt.tight_layout(); plt.show()

## 11. A10 — domestic demand is large, and flights do not drive it

- **We have** — the two domestic hotel files, with daily guests and daily arrivals.
- **The issue** — domestic guests are a large share of all guest nights and they mostly travel by
  road. Leave them out and the total is badly wrong. Drive them from flights and we invent a link.
- **We do** — add them as a control, not as a driver. Keep every aviation lever off this series.
- **New variables** — `dom_arrivals_7d`, `dom_guests_7d`.

In [ ]:
dom_daily = (dom.set_index("Date").reindex(panel_dates).rename_axis("Date").reset_index())
dom_daily["dom_guests_7d"]   = dom_daily["Guests"].rolling(7, min_periods=1).sum()
dom_daily["dom_arrivals_7d"] = dom_daily["New Arrivals"].rolling(7, min_periods=1).sum()
dom_daily = dom_daily[["Date", "dom_guests_7d", "dom_arrivals_7d"]]

# AVAILABILITY CHECK. The domestic test file drops `Guests` exactly as the international one
# does, so dom_guests_7d does not exist over the test window. A feature we cannot compute at
# prediction time is not a feature, however well it scores in training. Only dom_arrivals_7d
# survives into the model.
avail = (dom_daily.assign(test=dom_daily["Date"] > TUNE_END)
                  .groupby("test")[["dom_guests_7d", "dom_arrivals_7d"]]
                  .apply(lambda g: g.notna().mean().round(3)))
print("share of days where the value exists:")
print(avail.to_string())

# How large is domestic? Last 12 months of the train window.
yr = (dom["Date"] > TUNE_END - pd.DateOffset(years=1)) & (dom["Date"] <= TUNE_END)
d_sum = dom.loc[yr, "Guests"].sum()
i_sum = intl.loc[(intl["Date"] > TUNE_END - pd.DateOffset(years=1)) &
                 (intl["Date"] <= TUNE_END), "Guests"].sum()
print(f"last 12 train months -> domestic {d_sum:,.0f} guest nights, "
      f"international {i_sum:,.0f}  ({d_sum/(d_sum+i_sum):.1%} domestic)")
log("A10", f"domestic is {d_sum/(d_sum+i_sum):.1%} of guest nights; added as a control only")

## 12. A11 — the market grew across the window

- **We have** — four years of history.
- **The issue** — monthly seats nearly tripled across it. A model trained flat across the whole
  window learns a level that is too low for the test period.
- **We do** — add trend variables, and prefer a share to a level where a share works.
- **New variables** — `days_since_start`, `market_share_12m`.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.0))
monthly_seats = flights.groupby(flights["Date"].dt.to_period("M"))["Total Seats"].sum()
ax.plot(monthly_seats.index.to_timestamp(), monthly_seats.values / 1e6,
        color=BLUE, linewidth=2)
ax.axvspan(pd.Timestamp("2022-01-01"), pd.Timestamp("2022-12-31"),
           color=MUTED, alpha=0.12, linewidth=0)
ax.annotate("2022: monthly grain,\nexcluded from the panel", xy=(pd.Timestamp("2022-02-01"), 1.15),
            fontsize=8.5, color=INK2)
style(ax, "Monthly seats into Abu Dhabi: the window contains a strong recovery trend",
      ylabel="million seats")
plt.tight_layout(); plt.show()

## 13. Assemble the panel

One row per date and market. Every variable from steps A1 to A11, plus the target and the split
label. Steps A12's leakage rules are enforced here: we carry **no lagged target**, and
`New Arrivals` stays out of the feature list even though the test files supply it (it belongs to
Model R only, never Model S).

In [ ]:
panel = (fm_full
         .merge(hotel[["Date", "market", "Guests", "New Arrivals", "was_filled"]],
                on=["Date", "market"], how="left")
         .merge(cal, on="Date", how="left")
         .merge(dom_daily, on="Date", how="left")
         .merge(markets, on="market", how="left"))

panel["ym"] = panel["Date"].dt.to_period("M")
panel = panel.merge(wk, on=["ym", "market"], how="left")
panel["weekly_frequency"] = panel["weekly_frequency"].fillna(0)
panel = panel.merge(season_mkt, on=["market", "month"], how="left")

# Route age, per row rather than per market, so it moves through time.
panel["months_since_first_flight"] = (
    (panel["Date"] - panel["market"].map(first_flight)).dt.days / 30.44).round(1)
panel["route_is_new"] = (~panel["has_direct_route"]) | (
    panel["market"].map(first_flight).gt(FILE_START).fillna(False))

# Trend variables.
panel["days_since_start"] = (panel["Date"] - PANEL_START).dt.days
tot_7d = panel.groupby("Date")["p2p_7d"].transform("sum")
panel["market_share_12m"] = (panel["p2p_7d"] / tot_7d.replace(0, np.nan)).fillna(0)

# Split label.
panel["split"] = np.select(
    [panel["Date"] <= TRAIN_END, panel["Date"] <= TUNE_END],
    ["train", "tune"], default="test")

print(f"panel: {len(panel):,} rows x {panel.shape[1]} cols")
print(panel.groupby("split", sort=False).agg(rows=("Date", "size"),
      first=("Date", "min"), last=("Date", "max"),
      target_known=("Guests", lambda s: int(s.notna().sum()))).to_string())

In [ ]:
# Gates from plan_3 section 7. If any of these fail, nothing downstream is trustworthy.
expected_rows = len(panel_dates) * len(hotel_markets)
assert len(panel) == expected_rows, f"{len(panel)} rows, expected {expected_rows}"
assert panel["Date"].min() == PANEL_START and panel["Date"].max() == TEST_END
assert panel.groupby(["Date", "market"]).size().max() == 1, "duplicate date-market rows"
assert panel["load_factor_clean"].max() <= 1.0
assert not [c for c in panel.columns if "lag" in c.lower()], "a lagged target crept in"

# The target must be absent across the whole test window, and present everywhere it was
# supplied. A6 deliberately left a few completed grid cells missing, so those are the only
# allowed gaps before the test window.
assert panel.loc[panel["split"] == "test", "Guests"].isna().all(), "target leaked into test"
known = panel[panel["split"] != "test"]
unexplained = known["Guests"].isna() & ~known["was_filled"]
assert not unexplained.any(), f"{int(unexplained.sum())} target gaps on rows that WERE supplied"
print(f"all gates pass  ({int(known['Guests'].isna().sum())} target gaps, all on completed "
      f"grid cells that failed the A6 smallness check)")

In [ ]:
FEATURES = {
    "aviation supply": ["seats_7d", "flights_7d", "weekly_frequency", "seats_per_aircraft",
                        "load_factor_clean", "transfer_share",
                        "p2p_3d", "p2p_7d", "p2p_14d", "p2p_28d"],
    "route shape":     ["n_airlines", "n_cities", "route_is_new", "months_since_first_flight"],
    "market identity": ["market", "region", "haul_band", "capture_rate_hist",
                        "stay_length_hist", "elasticity_tier", "has_direct_route"],
    "season":          ["month", "day_of_week", "week_of_year", "season_index"],
    "events":          [c for c in panel.columns if c.startswith("is_")],
    # dom_guests_7d is deliberately absent: the domestic test file drops `Guests`, so the
    # value does not exist at prediction time. The column stays in the panel for analysis.
    "domestic control":["dom_arrivals_7d"],
    "trend":           ["days_since_start", "market_share_12m"],
}

rows = []
test_rows = panel[panel["split"] == "test"]
for group, cols in FEATURES.items():
    for c in cols:
        assert c in panel.columns, f"missing feature: {c}"
        rows.append({"group": group, "variable": c,
                     "nulls": int(panel[c].isna().sum()),
                     "null_pct_test": round(float(test_rows[c].isna().mean()), 3),
                     "dtype": str(panel[c].dtype)})
feature_table = pd.DataFrame(rows)
print(f"{len(feature_table)} model variables across {feature_table['group'].nunique()} groups")
feature_table

In [ ]:
# AVAILABILITY GATE. A feature that is mostly missing over the test window cannot be used,
# whatever it does in training. The two tolerated cases below are structural, not accidental.
TOLERATED = {
    "capture_rate_hist":         "the 12 markets with no direct route have no ratio to measure",
    "months_since_first_flight": "same 12 markets have no first flight",
    "load_factor_clean":         "a market with no flights in 7 days has no load factor",
    "transfer_share":            "same reason",
    "seats_per_aircraft":        "same reason",
}
problems = []
for _, r in feature_table.iterrows():
    if r["null_pct_test"] > 0.0 and r["variable"] not in TOLERATED:
        problems.append((r["variable"], r["null_pct_test"]))
    elif r["null_pct_test"] > 0.0:
        print(f"  tolerated: {r['variable']:26} {r['null_pct_test']:.1%} missing in test "
              f"- {TOLERATED[r['variable']]}")
assert not problems, f"features missing over the test window: {problems}"
print("\navailability gate passes")

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 3.2))
tgt = panel.groupby("Date")["Guests"].sum().replace(0, np.nan)
ax.plot(tgt.index, tgt.values / 1e3, color=BLUE, linewidth=1.2)
for (lo, hi, label, colour) in [
        (PANEL_START, TRAIN_END, "train", BLUE),
        (TRAIN_END, TUNE_END, "tune", AQUA),
        (TUNE_END, TEST_END, "test — target withheld", ORANGE)]:
    ax.axvspan(lo, hi, color=colour, alpha=0.08, linewidth=0)
    ax.annotate(label, xy=(lo + (hi - lo) / 2, 0.93), xycoords=("data", "axes fraction"),
                ha="center", fontsize=8.5, color=colour)
style(ax, "International guest nights, and the three windows",
      ylabel="thousand guest nights per day")
plt.tight_layout(); plt.show()

## 14. Save the outputs

In [ ]:
panel.to_parquet(FEAT_DIR / "panel.parquet", index=False)
fl.to_parquet(CLEAN_DIR / "flights.parquet", index=False)
hotel.to_parquet(CLEAN_DIR / "hotel_intl.parquet", index=False)
dom.to_parquet(CLEAN_DIR / "hotel_dom.parquet", index=False)
feature_table.to_csv(REPORT_DIR / "feature_table.csv", index=False)
elas.to_csv(REPORT_DIR / "elasticity_train.csv", index=False)

lines = ["# Cleaning log", "",
         f"Generated by `experiments/notebooks/01_cleaning_features.ipynb`.",
         f"Panel: {len(panel):,} rows, {panel['Date'].min():%Y-%m-%d} to "
         f"{panel['Date'].max():%Y-%m-%d}, {panel['market'].nunique()} markets.", ""]
for r in LOG:
    lines.append(f"- **{r['step']}** — {r['note']}")
(REPORT_DIR / "cleaning_log.md").write_text("\n".join(lines) + "\n")

for p in [FEAT_DIR / "panel.parquet", CLEAN_DIR / "flights.parquet",
          CLEAN_DIR / "hotel_intl.parquet", CLEAN_DIR / "hotel_dom.parquet",
          REF_DIR / "bridge.csv", REF_DIR / "events.csv", REF_DIR / "markets.csv",
          REPORT_DIR / "cleaning_log.md", REPORT_DIR / "feature_table.csv",
          REPORT_DIR / "elasticity_train.csv"]:
    print(f"  {p.relative_to(PROJECT)}  ({p.stat().st_size/1024:,.0f} KB)")

## What is not done here

These belong to the modelling notebook, not to cleaning:

1. **The indirect bridge allocation.** `reference/bridge.csv` marks the 12 markets with no direct
   route as `pending`. Splitting transfer flows onto nationalities is step B5 of `plan_3.md`.
2. **`reference/cities.csv`.** `haul_band` is derived from `region` as a stated approximation. A
   true distance band needs a coordinate per departure city.
3. **Event dates need confirming.** Every row of `reference/events.csv` carries
   `estimate - confirm with DCT`.
4. **Occupancy.** The dataset holds no room supply count, so the target is guest nights.